# Lesson 10 — Pathlib

Paths as objects, not strings. Join, inspect, list, read, write.

## How to work

- Write every solution in `solution.py` first, then paste the finished
  code into the matching cell and run it.
- **Rule:** your printed output must match the `Output:` block exactly.
  File contents are output too.
- **Constraint = checklist.** Before you code, read the Constraint lines
  and treat each one as a checkbox to honor. A missed constraint is a
  fail even if the output looks right.
- Every cell builds its own fixture folders and files first — cells are
  self-contained; re-running a cell reproduces its output exactly.
- Every `Path` read/write passes `encoding="utf-8"` unless a constraint
  says otherwise.
- Functions never print unless the task says so; printing happens in
  the driver code.
- Hints are conceptual keywords, not answers — ask for one at a time.
- H1 ships broken code on purpose: find and fix the bugs; don't rewrite
  working parts.

Difficulty climbs after the Easy set.

## E1 — Inspect a path

**Type:** [REAL-WORLD]

**Task:** write `describe(path)` that returns a tuple
`(name, stem, suffix, parent_stem)`. For the given path, print the four
values. Driver:

```python
p = Path("reports/2026/monthly/sales.csv")
name, stem, suffix, parent_stem = describe(p)
print(name)
print(stem)
print(suffix)
print(parent_stem)
```

**Output:**
```
sales.csv
sales
.csv
monthly
```

**Constraint (checklist):**
- Use `/.name`, `/.stem`, `/.suffix` on the path object.
- For the parent, take `.parent` then `.name` to get the enclosing folder's name.
- `describe` returns data; it never prints.

**Hints (ask one at a time):**
- `Path("a/b/x.txt").stem` is `x`; what is `.suffix`?
- `.parent` of that path is `a/b` — how do you get just the last folder's name?
- The driver prints four separate lines in order.

In [6]:
from pathlib import Path

def describe(path: Path):
    return path.name, path.stem, path.suffix, path.parent.name

p = Path("reports/2026/monthly/sales.csv")
name, stem, suffix, parent_stem = describe(p)
print(name)
print(stem)
print(suffix)
print(parent_stem)

sales.csv
sales
.csv
monthly


## E2 — Join with /

**Type:** [REAL-WORLD]

**Task:** write `data_file(base, year, name)` that takes a base folder
(`str`), a year (`int`), and a file name (`str`) and returns the joined
path. Driver:

```python
p = data_file("data", 2026, "expenses.txt")
print(p)
print(p.parent)
print(p.name)
```

**Output:**
```
data\2026\expenses.txt
data\2026
expenses.txt
```

**Constraint (checklist):**
- Build the path with `/` (start with `Path(base)`).
- `str(year)` before joining — `/` wants path parts, not numbers.
- `data_file` returns the `Path`; it never prints.

**Example:**
`data_file("data", 2026, "expenses.txt")` → a `Path` that prints as
`data\2026\expenses.txt` on Windows.

**Hints (ask one at a time):**
- `/` works left to right: `Path("data") / 2026 / name` — but 2026 is an int.
- Convert the int to `str` before it touches `/`.
- The slashes in the Output are your OS separators — `Path` adds them for you.

In [10]:
def data_file(base, year, name):
    year = str(year)
    return Path(base) / year / name

p = data_file("data", 2026, "expenses.txt")
print(p)
print(p.parent)
print(p.name)

<class 'pathlib.WindowsPath'>
data\2026
expenses.txt


## E3 — Does it exist?

**Type:** [REAL-WORLD]

**Task:** write `status(path)` returning `"exists"` if the path exists,
otherwise `"missing"`. Driver:

```python
Path("present.txt").write_text("x", encoding="utf-8")
print(status(Path("present.txt")))
print(status(Path("absent.txt")))
```

**Output:**
```
exists
missing
```

**Constraint (checklist):**
- Use `.exists()` on the path — a bool.
- `status` returns the strings exactly as shown; it never prints.
- No try/except here — `.exists()` answers before any risky operation.

**Hints (ask one at a time):**
- `Path.exists()` returns `True` or `False`.
- Turn that bool into the two allowed strings.
- You are choosing LBYL (check first), not EAFP — Lesson 8's decision rule.

In [13]:
def status(path: Path):
    return "exists" if path.exists() else "missing"

Path("present.txt").write_text("x", encoding="utf-8")
print(status(Path("present.txt")))
print(status(Path("absent.txt")))

exists
missing


## E4 — List only .txt files

**Type:** [REAL-WORLD]

**Task:** write `list_txt(folder)` returning a sorted list of the NAMES
of the `.txt` files inside a folder (names only, not full paths). Driver:

```python
folder = Path("docs")
folder.mkdir(exist_ok=True)
for name in ["a.txt", "b.csv", "c.txt", "d.log"]:
    (folder / name).write_text("hi", encoding="utf-8")

print(list_txt(folder))
```

**Output:**
```
['a.txt', 'c.txt']
```

**Constraint (checklist):**
- Use `.iterdir()` to get the entries.
- Filter with `.suffix == ".txt"`.
- Collect `.name`, not the full child path.
- Return a `sorted` list. `list_txt` never prints.

**Hints (ask one at a time):**
- `folder.iterdir()` yields each child — check one child's `.suffix`.
- Build a list of `.name` for the matching children.
- `sorted` gives deterministic order, matching the output.

In [28]:
def list_txt(folder: Path):
    names = []
    for file in folder.iterdir():
        if file.suffix == ".txt":
            names.append(file.name)
    return names
    # i have a question
    # as i tested with multiple file name variation
    # the result is always sorted even without using sorted(names)
    # so it is unnecessary to add sorted() for the result right?
        
folder = Path("docs")
folder.mkdir(exist_ok=True)
for name in ["c.txt", "b.csv", "a.txt", "d.log"]:
    (folder / name).write_text("hi", encoding="utf-8")

print(list_txt(folder))

['a.txt', 'c.txt']


## E5 — Whole-file round trip

**Type:** [REAL-WORLD]

**Task:** write `save_text(path, content)` using `write_text`, and
`load_text(path)` using `read_text` (both return/save data; note
`save_text` returns `True` when done). Run the driver: write two lines,
read them back, then write again and read again. Driver:

```python
p = Path("story.txt")
print(save_text(p, "line one\nline two\n"))
print(repr(load_text(p)))
print(save_text(p, "replacement\n"))
print(repr(load_text(p)))
print(repr(load_text(p)))
```

**Output:**
```
True
'line one\nline two\n'
True
'replacement\n'
'replacement\n'
```

**Constraint (checklist):**
- `save_text` uses `write_text` with `encoding="utf-8"` and returns `True`.
- `load_text` uses `read_text` with `encoding="utf-8"` and returns the string.
- Both functions never print.

**Example:**
writing `"replacement\n"` over the file erases `"line one\nline two\n"` —
`write_text` rewrites the whole file, like `"w"` mode.

**Hints (ask one at a time):**
- `write_text` returns `None`; your function must return `True`.
- `repr(s)` shows the `\n` characters as text instead of new lines.
- The second write replaces the first — read again to confirm.

In [31]:
def save_text(path: Path, content):
    path.write_text(content, encoding="utf-8")
    return True

def load_text(path: Path):
    return path.read_text(encoding="utf-8")
    
    
p = Path("story.txt")
print(save_text(p, "line one\nline two\n"))
print(repr(load_text(p)))
print(save_text(p, "replacement\n"))
print(repr(load_text(p)))
print(repr(load_text(p)))

True
'line one\nline two\n'
True
'replacement\n'
'replacement\n'


## M1 — read_text summary

**Type:** [REAL-WORLD]

**Task:** a file holds one `name,score` per line. Write
`summarize(path)` returning a dict with `"total"` (sum of valid scores)
and `"valid"` (count of valid lines). Use `read_text()` to load the
whole file, then process the string. Driver:

```python
Path("scores.txt").write_text("ana,10\nbob,20\nbad\ncid,5\n", encoding="utf-8")

s = summarize(Path("scores.txt"))
print(f"total: {s['total']}")
print(f"valid: {s['valid']}")
```

**Output:**
```
total: 35
valid: 3
```

**Constraint (checklist):**
- Load with `read_text(encoding="utf-8")` — a `Path` method, not `open`.
- Split the loaded string into lines with `.split("\n")` (or `splitlines`).
- A valid line is `name,value` with `value` convertible to `int`. Invalid = skipped (not an error).
- Wrap the risky conversion in `try/except ValueError` (Lesson 8).
- `summarize` returns the dict; it never prints.

**Hints (ask one at a time):**
- `read_text()` gives you one big string — split it into lines first.
- You did this exact validation in Lesson 9's M1 — only the load line changed.
- Count and total start at 0 before any line is processed.

In [45]:
def summarize(path: Path):
    summary = {
        "total": 0,
        "valid": 0,
    }
    lines = path.read_text(encoding="utf-8")
    for line in lines.splitlines():
        if "," not in line:
            continue
        try:
            name, value = line.split(",")
            value = int(value)
        except:
            continue
        else:
            summary["total"] += value
            summary["valid"] += 1
    return summary
    
    
Path("scores.txt").write_text("ana,10\nbob,20\nbad\ncid,5\n", encoding="utf-8")

s = summarize(Path("scores.txt"))
print(f"total: {s['total']}")
print(f"valid: {s['valid']}")

total: 35
valid: 3


## M2 — Organize by extension

**Type:** [REAL-WORLD]

**Task:** a folder is full of files. Write `organize(folder)` that creates
one subfolder per unique extension (named without the dot, e.g. `csv`),
moves each file into its match, and returns a sorted list of
`extension: count` strings. Driver:

```python
folder = Path("inbox")
folder.mkdir(exist_ok=True)
for f in ["a.csv", "b.txt", "c.csv"]:
    (folder / f).write_text("x", encoding="utf-8")

for line in organize(folder):
    print(line)

print(sorted(p.name for p in folder.iterdir() if p.is_dir()))
```

**Output:**
```
csv: 2
txt: 1
['csv', 'txt']
```

**Constraint (checklist):**
- For each entry, use `mkdir(exist_ok=True)` on the target subfolder before moving.
- Move with `.rename()` (target = folder / file.name).
- Only move FILES, not folders — check `.is_dir()`.
- Return `sorted` `"suffix-without-dot: count"` lines. `organize` never prints.
- Re-running `organize` must not crash (exist_ok) — idempotent.

**Example:**
`a.csv` and `c.csv` move into folder `csv/`, so `csv: 2`. Folder `txt/`
holds `b.txt`, so `txt: 1`.

**Hints (ask one at a time):**
- `.suffix` gives `".csv"` — slice off the dot for the folder name.
- `mkdir(exist_ok=True)` makes the folder once; a second call is fine.
- Count with the Lesson 6 dict-counting pattern, then sort the result.

In [76]:
def organize(folder: Path):
    result = {}
    
    for file in folder.iterdir():
        if file.is_dir():
            continue
        
        ext = file.suffix[1:] 
        subfolder = folder / ext
        subfolder.mkdir(exist_ok=True)
        
        target = subfolder / file.name
        if target.exists():
            continue
        
        file.rename(target)
        result[ext] = result.get(ext, 0) + 1
        
    return sorted(f"{key}: {value}" for key, value in result.items())

    
folder = Path("inbox")
folder.mkdir(exist_ok=True)
for f in ["a.csv", "b.txt", "c.csv"]:
    (folder / f).write_text("x", encoding="utf-8")

for line in organize(folder):
    print(line)

print(sorted(p.name for p in folder.iterdir() if p.is_dir()))

csv: 2
txt: 1
['csv', 'txt']


## M3 — Typed loader on Path

**Type:** [REAL-WORLD]

**Task:** config files hold `key=value` lines. Your program wants typed
values. Given `SPEC = {"retries": int, "rate": float, "name": str}`,
write `load_settings(path)` that reads with `Path.read_text()`, processes
each non-blank line (split on the FIRST `=`), and — for keys in SPEC —
stores the converted value. Unknown keys are ignored. The result starts
EMPTY `{}`. Conversion failure raises
`ValueError(f"bad setting: {key}")`. Driver:

```python
SPEC = {"retries": int, "rate": float, "name": str}
Path("app.cfg").write_text("retries=3\nrate=0.5\nname=cli\n", encoding="utf-8")
print(load_settings(Path("app.cfg")))

Path("bad.cfg").write_text("retries=three\nrate=0.5\n", encoding="utf-8")
try:
    print(load_settings(Path("bad.cfg")))
except ValueError as err:
    print(err)
```

**Output:**
```
{'retries': 3, 'rate': 0.5, 'name': 'cli'}
bad setting: retries
```

**Constraint (checklist):**
- Load with `read_text(encoding="utf-8")`; split the string into lines.
- Strip whitespace from each line and from the VALUE before converting.
- Start the result EMPTY `{}`; add a key only when the file contains it.
- Unknown keys are ignored (not stored, not errors).
- Conversion failure raises the exact message `bad setting: <key>`.

**Example:**
`int("3\n")` still works, but `str("cli\n")` keeps the `\n` — so strip the
value first. This is the Lesson 9 gotcha in a new wrapper.

**Hints (ask one at a time):**
- This is Lesson 9's M3 with `read_text` instead of `for line in f`.
- `read_text()` gives one string — split it into lines before the loop.
- Raise keeps the key name so the caller knows which setting was bad.

In [4]:
def load_settings(path: Path):
    result = {}
    
    lines = path.read_text(encoding="utf-8")
    for line in lines.split():
        key, value = line.strip().split("=", 1)
        if key in SPEC:
            try:
                result[key] = SPEC[key](value)
            except ValueError:
                raise ValueError(f"bad setting: {key}") 
    return result


SPEC = {"retries": int, "rate": float, "name": str}
Path("app.cfg").write_text("retries=3\nrate=0.5\nname=cli\n", encoding="utf-8")
print(load_settings(Path("app.cfg")))

Path("bad.cfg").write_text("retries=three\nrate=0.5\n", encoding="utf-8")
try:
    print(load_settings(Path("bad.cfg")))
except ValueError as err:
    print(err)

{'retries': 3, 'rate': 0.5, 'name': 'cli'}
bad setting: retries


## H1 — Debug: three bugs in the folder tool

**Type:** [REAL-WORLD] · debug drill

`report_folder` was written carelessly. It should list the base names of
the `.py` **files** in a folder. Run it, compare expected vs observed, and
fix **three** bugs.

**Expected output (after fixes):**
```
['app.py', 'util.py']
True
```

**Observed as-is:** an empty list and `False`. That is a clue, not a
crash — the filter matches nothing. Then the next bug appears as you fix
each one. Read the driver: it creates real `.py` files **and a folder
called `data.py`**.

**Rules:**
- Fix each bug with the MINIMAL change. No rewriting working parts.
- Do not modify the driver block.
- After fixing, explain each bug: expected vs observed, root cause.

**The three bug families:**
1. The suffix filter uses the wrong case.
2. The names are uppercased — `APP.PY` instead of `app.py`.
3. `iterdir()` yields folders too — the result wrongly includes the
   `data.py` folder. Guard it.

**Hints (ask one at a time):**
- Compare the suffixes you filter against what the test files use.
- `.upper()` is a string method — what did the tool add to the names?
- The driver creates a folder named `data.py`. Does your loop treat it
  like a file?

In [ ]:
from pathlib import Path

def report_folder(folder):
    result = []
    for child in folder.iterdir():
        if child.suffix == ".py" and not child.is_dir(): # change .PY to .py, add not is_dir
            result.append(child.name) # remove upper()
    return result


# --- driver (do not change) ---
folder = Path("proj")
folder.mkdir(exist_ok=True)
for name in ["app.py", "util.py", "readme.txt"]:
    (folder / name).write_text("x", encoding="utf-8")
(folder / "data.py").mkdir()

print(report_folder(folder))
print(report_folder(folder) == ["app.py", "util.py"])

['app.py', 'util.py']
True


In [ ]:
from pathlib import Path

def report_folder(folder):
    result = []
    for child in folder.iterdir():
        if child.suffix == ".PY":
            result.append(child.name.upper())
    return result


# --- driver (do not change) ---
folder = Path("proj")
folder.mkdir(exist_ok=True)
for name in ["app.py", "util.py", "readme.txt"]:
    (folder / name).write_text("x", encoding="utf-8")
(folder / "data.py").mkdir()

print(report_folder(folder))
print(report_folder(folder) == ["app.py", "util.py"])

## H2 — Interview drill: count by extension

**Type:** [INTERVIEW]

**Task:** implement `count_ext(folder)` returning a dict mapping each
extension (with the dot, e.g. `".csv"`) to how many files have it. Only
COUNT FILES (not folders). Driver:

```python
folder = Path("mix")
folder.mkdir(exist_ok=True)
for f in ["a.csv", "b.txt", "c.csv", "d.py", "e.csv", "notes.txt"]:
    (folder / f).write_text("x", encoding="utf-8")
(folder / "sub").mkdir(exist_ok=True)

print(count_ext(folder))
print("empty:", count_ext(Path("emptydir")))
```

**Output:**
```
{'.csv': 3, '.txt': 2, '.py': 1}
empty: {}
```

**Constraint (checklist):**
- Use `.iterdir()` once; skip folders with `.is_dir()`.
- Key each extension as shown, WITH the dot.
- Count with the Lesson 6 `dict.get` pattern.
- An empty folder returns `{}` — no special error, just no counts.

**Hints (ask one at a time):**
- `dict.get(suffix, 0)` counts without a KeyError.
- Sorting is not required — dict order is insertion order (CSV first here).
- The empty folder has no entries to iterate, so the dict stays empty.

In [30]:
def count_ext(path: Path):
    result = {}
    
    if not path.exists():
        return result
    
    for item in path.iterdir():
        if not item.is_file():
            continue
        
        ext = item.suffix
        result[ext] = result.get(ext, 0) + 1
        
    return result

folder = Path("mix")
folder.mkdir(exist_ok=True)
for f in ["a.csv", "b.txt", "c.csv", "d.py", "e.csv", "notes.txt"]:
    (folder / f).write_text("x", encoding="utf-8")
(folder / "sub").mkdir(exist_ok=True)

print(count_ext(folder))
print("empty:", count_ext(Path("emptydir")))

{'.csv': 3, '.txt': 2, '.py': 1}
empty: {}


## Challenge — Folder Reporter

**Type:** [REAL-WORLD] · micro build

Build a small tool that reports on a folder of files and writes its
report to a text file. Minimum three functions:

- `count_by_ext(folder)` — a dict: extension (with dot) → count of files
  with that extension. Only files, not folders.
- `largest(folder)` — the NAME of the largest file (by size, `.stat().st_size`)
  or `None` when the folder is empty.
- `write_report(folder, out_path)` — writes lines to `out_path` using
  `Path.write_text` and RETURNS the report as a string. Line format below.

Report format (one per line):
```
Extension summary:
<ext>: <count>
...
Largest file: <name>
```

Driver:

```python
folder = Path("assets")
folder.mkdir(exist_ok=True)
sizes = {"aa.csv": 100, "bb.txt": 300, "cc.csv": 200, "dd.log": 50}
for name, size in sizes.items():
    (folder / name).write_bytes(b"x" * size)

body = write_report(folder, Path("assets_report.txt"))
print(body, end="")
print("--- file ---")
print(Path("assets_report.txt").read_text(encoding="utf-8"), end="")
print("empty:", largest(Path("empty_assets")))
```

**Output:**
```
Extension summary:
.csv: 2
.log: 1
.txt: 1
Largest file: bb.txt
--- file ---
Extension summary:
.csv: 2
.log: 1
.txt: 1
Largest file: bb.txt
empty: None
```

**Constraints (checklist):**
- `count_by_ext` counts only files (skip `.is_dir()`). Key with the dot.
- `largest` uses `.stat().st_size` and returns the NAME (string), or `None`.
- `write_report` uses `Path.write_text(encoding="utf-8")` for the file, builds
  the report string first, and returns that same string.
- Extension lines are sorted alphabetically for deterministic output.
- The report string ends with a trailing `\n` (after `Largest file: <name>`);
  the file content must equal the returned report, so both end with `\n`.

**Acceptance criteria:** runs clean on the given input AND on an empty
folder; the file content equals the returned report; re-running gives
the same output (idempotent).

**Stretch goals (optional):**
- Add `Total size: <N>` and `Files: <N>` lines to the report.
- Make `write_report` also accept an injectable output writer `out=write_file`
  so tests can capture without touching disk.
- Skip hidden files (names starting with a dot) in the count.

**Hints (ask one at a time):**
- Data flow first: counts from the folder, largest from the folder, then build
  the string, then write it once.
- `sorted()` the extension lines so the order is stable.
- `largest` falls back to `None` when there is nothing to compare — same
  sentinel idea as your very first min/max lesson.

In [34]:
def count_by_ext(folder: Path):
    result = {}
    if not folder.exists():
        return result
    
    for item in folder.iterdir():
        if not item.is_file():
            continue
        
        ext = item.suffix
        result[ext] = result.get(ext, 0) + 1
        
    return result

def get_size(file_path: Path):
    return file_path.stat().st_size

def largest(folder: Path):
    if not folder.exists():
        return None
    
    files = [item for item in folder.iterdir() if item.is_file()]
    
    largest_file = max(files, key=get_size)
    
    return largest_file.name

def write_report(folder: Path, out_path: Path):
    ext_counts = count_by_ext(folder)
    largest_file_name = largest(folder)
    
    report_lines = ["Extension summary:"]
    
    for key, value in sorted(ext_counts.items()):
        report_lines.append(f"{key}: {value}")
    
    report_lines.append(f"Largest file: {largest_file_name}")
    
    report_string = "\n".join(report_lines) + "\n"
    out_path.write_text(report_string, encoding="utf-8")
    return report_string


folder = Path("assets")
folder.mkdir(exist_ok=True)
sizes = {"aa.csv": 100, "bb.txt": 300, "cc.csv": 200, "dd.log": 50}
for name, size in sizes.items():
    (folder / name).write_bytes(b"x" * size)

body = write_report(folder, Path("assets_report.txt"))
print(body, end="")
print("--- file ---")
print(Path("assets_report.txt").read_text(encoding="utf-8"), end="")
print("empty:", largest(Path("empty_assets")))

Extension summary:
.csv: 2
.log: 1
.txt: 1
Largest file: bb.txt
--- file ---
Extension summary:
.csv: 2
.log: 1
.txt: 1
Largest file: bb.txt
empty: None
